# Phase 04P: Teacher C → S3 Logit-KD Training Readiness

Notebook khóa production engine trước full run seed 42:

- Teacher C SigLIP ViT-B/16 frozen;
- Student S3 TinyViT-5M trainable;
- Direct Logit-KD, `T=4`, `alpha=0.5`;
- image-balanced Train+Restval, Validation-only selection;
- AdamW, warmup-cosine, FP16 GradScaler;
- teacher-forced Validation, Beam-3 và COCO metrics;
- atomic checkpoint, discarded four-step pilot;
- Held-out Test luôn `SEALED`.


## Cell 1 — Authorization, environment và integrity

Xác minh Phase 04O, Teacher C seed 42, Teacher Pool, Phase 02A, offline
SigLIP và toàn bộ checksum. Chưa đọc dataset hoặc deserialize checkpoint.


In [1]:
import copy
import csv
import hashlib
import json
import math
import os
import platform
import random
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch


CELL_STARTED_AT = time.perf_counter()
SEED = 42
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path(
    "/kaggle/working//"
    "phase4_teacher_c_s3_logit_kd_training_readiness"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
LOG_DIR = OUTPUT_ROOT / "logs"
RESULT_DIR = OUTPUT_ROOT / "results"
VALIDATION_DIR = OUTPUT_ROOT / "validation"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT, AUDIT_DIR, CONFIG_DIR, LOG_DIR, RESULT_DIR,
    VALIDATION_DIR, CHECKPOINT_DIR, MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def exactly_one(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, target)
    return target


def set_deterministic(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)


def verify_manifest(root, csv_path, sums_path):
    rows = list(csv.DictReader(
        Path(csv_path).open("r", encoding="utf-8", newline="")
    ))
    expected = []
    for row in rows:
        artifact = resolve_inside(root, row["relative_path"])
        assert artifact.is_file()
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256_file(artifact) == row["sha256"]
        expected.append(f'{row["sha256"]}  {row["relative_path"]}')
    actual = [
        line.strip()
        for line in Path(sums_path).read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]
    assert actual == expected
    return rows


if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA không khả dụng. Phase 04P yêu cầu GPU T4 hoặc mạnh hơn."
    )
set_deterministic(SEED)
device = torch.device("cuda")

PHASE4O_COMPLETE = exactly_one(
    "PHASE4_TEACHER_C_S3_LOGIT_KD_PREFLIGHT_COMPLETE.json"
)
PHASE4P_AUTHORIZED = exactly_one(
    "PHASE4_TEACHER_C_S3_LOGIT_KD_"
    "TRAINING_READINESS_AUTHORIZED.json"
)
PHASE2_READY = exactly_one(
    "PHASE2_SHARED_TRAINING_BUNDLE_READY.json"
)
PHASE1_READY = exactly_one(
    "PHASE1_FORMAL_DELIVERABLES_READY.json"
)
TEACHER_RUN_COMPLETE = exactly_one(
    "TEACHER_C_SEED42_FULL_RUN_COMPLETE.json"
)
TEACHER_POOL_COMPLETE = exactly_one(
    "PHASE3_TEACHER_POOL_COMPLETE.json"
)

phase4o_complete = read_json(PHASE4O_COMPLETE)
phase4p_authorized = read_json(PHASE4P_AUTHORIZED)
phase2_ready = read_json(PHASE2_READY)
teacher_run = read_json(TEACHER_RUN_COMPLETE)
teacher_pool = read_json(TEACHER_POOL_COMPLETE)

assert phase4o_complete["status"] == "complete"
assert phase4o_complete["checkpoint_strict_loads_passed"] == 5
assert phase4o_complete["loss_semantics_passed"] is True
assert phase4o_complete["forward_backward_passed"] is True
assert phase4o_complete["one_batch_overfit_passed"] is True
assert phase4o_complete["exact_resume_passed"] is True
assert phase4o_complete[
    "phase04p_training_readiness_authorized"
] is True
assert phase4p_authorized["status"] == "authorized"
assert phase4p_authorized["experiment_id"] == "P4C_C_S3_LOGIT"
assert phase4p_authorized["teacher_id"] == "C"
assert phase4p_authorized["student_id"] == "S3"
assert phase4p_authorized["kd_channel"] == "direct_logit"
assert float(phase4p_authorized["temperature"]) == 4.0
assert float(phase4p_authorized["alpha"]) == 0.5
assert phase2_ready["status"] == "ready"
assert teacher_run["status"] == "complete"
assert teacher_run["teacher_id"] == "C"
assert teacher_run["qualified_for_kd"] is True
assert teacher_pool["status"] == "complete"
assert teacher_pool["all_teacher_runs_qualified_for_kd"] is True

for marker in [
    phase4o_complete, phase4p_authorized,
    teacher_run, teacher_pool,
]:
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False

PHASE4O_ROOT = PHASE4O_COMPLETE.parent
authorization_ref = phase4o_complete["authorization_marker"]
assert PHASE4P_AUTHORIZED == resolve_inside(
    PHASE4O_ROOT, authorization_ref["relative_path"]
)
assert sha256_file(PHASE4P_AUTHORIZED) == (
    authorization_ref["sha256"]
)
verified_loss_ref = phase4p_authorized["verified_loss_contract"]
VERIFIED_LOSS_CONTRACT = resolve_inside(
    PHASE4O_ROOT, verified_loss_ref["relative_path"]
)
assert sha256_file(VERIFIED_LOSS_CONTRACT) == (
    verified_loss_ref["sha256"]
)

phase4o_manifest = resolve_inside(
    PHASE4O_ROOT,
    phase4o_complete["manifest"]["csv"]["relative_path"],
)
phase4o_sums = resolve_inside(
    PHASE4O_ROOT,
    phase4o_complete["manifest"]["sha256sums"]["relative_path"],
)
assert sha256_file(phase4o_manifest) == (
    phase4o_complete["manifest"]["csv"]["sha256"]
)
assert sha256_file(phase4o_sums) == (
    phase4o_complete["manifest"]["sha256sums"]["sha256"]
)
phase4o_rows = verify_manifest(
    PHASE4O_ROOT, phase4o_manifest, phase4o_sums
)
assert len(phase4o_rows) == 11

PHASE2_ROOT = PHASE2_READY.parent
phase2_manifest = resolve_inside(
    PHASE2_ROOT, phase2_ready["manifests"]["csv"]["relative_path"]
)
phase2_sums = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["manifests"]["sha256sums"]["relative_path"],
)
assert sha256_file(phase2_manifest) == (
    phase2_ready["manifests"]["csv"]["sha256"]
)
assert sha256_file(phase2_sums) == (
    phase2_ready["manifests"]["sha256sums"]["sha256"]
)
phase2_rows = verify_manifest(
    PHASE2_ROOT, phase2_manifest, phase2_sums
)
assert len(phase2_rows) == 38
phase2_by_path = {row["relative_path"]: row for row in phase2_rows}

S3_INITIAL_CHECKPOINT = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["required_artifacts"]["s3_encoder_checkpoint"],
)
DECODER_INITIAL_CHECKPOINT = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["required_artifacts"]["shared_decoder_checkpoint"],
)
for checkpoint in [
    S3_INITIAL_CHECKPOINT, DECODER_INITIAL_CHECKPOINT,
]:
    relative = checkpoint.relative_to(PHASE2_ROOT).as_posix()
    assert sha256_file(checkpoint) == phase2_by_path[relative]["sha256"]
DECODER_CONTRACT = exactly_one(
    "shared_transformer_decoder_contract.json"
)
SHARED_DECODER_CONTRACT = DECODER_CONTRACT
SHARED_DECODER_INITIAL_CHECKPOINT = DECODER_INITIAL_CHECKPOINT

teacher_pool_root = TEACHER_POOL_COMPLETE.parent
teacher_registry_path = resolve_inside(
    teacher_pool_root,
    teacher_pool["teacher_c_checkpoint_registry"]["relative_path"],
)
assert sha256_file(teacher_registry_path) == (
    teacher_pool["teacher_c_checkpoint_registry"]["sha256"]
)
teacher_registry = read_json(teacher_registry_path)
seed42_registry = [
    row for row in teacher_registry["entries"]
    if int(row["seed"]) == SEED
][0]
TEACHER_CHECKPOINT = exactly_one("teacher_c_seed42_best.pt")
assert sha256_file(TEACHER_CHECKPOINT) == (
    seed42_registry["best_checkpoint"]["sha256"]
)
TEACHER_C_BEST_CHECKPOINT = TEACHER_CHECKPOINT

siglip_manifests = []
for source_path in INPUT_ROOT.rglob("MODEL_SOURCE_MANIFEST.json"):
    try:
        source_manifest = read_json(source_path)
    except Exception:
        continue
    if source_manifest.get("repo_id") == (
        "google/siglip-base-patch16-224"
    ):
        siglip_manifests.append(
            (source_path.resolve(), source_manifest)
        )
assert len(siglip_manifests) == 1, siglip_manifests
SIGLIP_SOURCE_MANIFEST_PATH, SIGLIP_SOURCE_MANIFEST = (
    siglip_manifests[0]
)
SIGLIP_SOURCE_DIR = SIGLIP_SOURCE_MANIFEST_PATH.parent
assert SIGLIP_SOURCE_MANIFEST["resolved_commit"] == (
    "7fd15f0689c79d79e38b1c2e2e2370a7bf2761ed"
)
for row in SIGLIP_SOURCE_MANIFEST["files"]:
    artifact = resolve_inside(
        SIGLIP_SOURCE_DIR, row["relative_path"]
    )
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

write_json(
    AUDIT_DIR / "phase4p_cell01_authorization.json",
    {
        "audit_version": "phase4p_cell01_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "phase4o_manifest_rows": len(phase4o_rows),
        "phase2_manifest_rows": len(phase2_rows),
        "strict_deterministic": True,
        "temperature": 4.0,
        "alpha": 0.5,
        "teacher_checkpoint_sha256": sha256_file(
            TEACHER_CHECKPOINT
        ),
        "s3_checkpoint_sha256": sha256_file(
            S3_INITIAL_CHECKPOINT
        ),
        "siglip_source_verified": True,
        "dataset_read": False,
        "caption_read": False,
        "checkpoint_payload_read": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04P CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Phase 04O manifest        : 11 / 11 VERIFIED")
print("Phase 02A manifest        : 38 / 38 VERIFIED")
print("Preflight gates           : ALL PASSED")
print("Phase 04P authorization   : VERIFIED")
print("Teacher C checkpoint      : VERIFIED")
print("S3 initialization         : VERIFIED")
print("Offline SigLIP            : VERIFIED")
print("Temperature / alpha       : 4.0 / 0.5")
print("GPU                       :", torch.cuda.get_device_name(0))
print("Strict deterministic      : True")
print("Checkpoint payload read   : False")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Next                      : production datasets and loaders")


PHASE 04P CELL 1 STATUS
Status                    : PASSED
Phase 04O manifest        : 11 / 11 VERIFIED
Phase 02A manifest        : 38 / 38 VERIFIED
Preflight gates           : ALL PASSED
Phase 04P authorization   : VERIFIED
Teacher C checkpoint      : VERIFIED
S3 initialization         : VERIFIED
Offline SigLIP            : VERIFIED
Temperature / alpha       : 4.0 / 0.5
GPU                       : Tesla T4
Strict deterministic      : True
Checkpoint payload read   : False
Full training started     : False
Test Set status           : SEALED
Next                      : production datasets and loaders


## Cell 2 — Production datasets, shared geometry và sampler

Khóa 113.287 ảnh Train+Restval, một caption cyclic/ảnh/epoch, shared
stateless RandomResizedCrop/flip, SigLIP normalization cho Teacher và
ImageNet normalization cho S3. Validation gồm 25.010 caption/5.000 ảnh.


In [2]:
CELL_STARTED_AT = time.perf_counter()
from collections import defaultdict

import pyarrow.parquet as pq
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF


TRAIN_PARQUET = exactly_one("mscoco_train.parquet")
VALIDATION_PARQUET = exactly_one("mscoco_validation.parquet")
IMAGE_MANIFEST = exactly_one("mscoco_image_manifest.parquet")


def find_image_leaf(name, prefix):
    matches = []
    for path in INPUT_ROOT.rglob(name):
        if path.is_dir() and next(path.glob(f"{prefix}*.jpg"), None):
            matches.append(path.resolve())
    matches = sorted(set(matches))
    assert len(matches) == 1, matches
    return matches[0]


TRAIN_LEAF = find_image_leaf("train2014", "COCO_train2014_")
VAL_LEAF = find_image_leaf("val2014", "COCO_val2014_")

path_table = pq.read_table(
    IMAGE_MANIFEST,
    columns=[
        "image_id", "experiment_split", "source_partition", "image_path"
    ],
    filters=[
        ("experiment_split", "in", ["train", "validation"])
    ],
)
TRAIN_PATHS = {}
VALIDATION_PATHS = {}
for row in path_table.to_pylist():
    image_id = int(row["image_id"])
    leaf = (
        TRAIN_LEAF
        if row["source_partition"] == "train2014"
        else VAL_LEAF
    )
    path = leaf / Path(row["image_path"]).name
    target = (
        TRAIN_PATHS
        if row["experiment_split"] == "train"
        else VALIDATION_PATHS
    )
    target[image_id] = path
assert len(TRAIN_PATHS) == 113287
assert len(VALIDATION_PATHS) == 5000

TRAIN_CAPTIONS = defaultdict(list)
for batch_arrow in pq.ParquetFile(TRAIN_PARQUET).iter_batches(
    batch_size=16384,
    columns=["annotation_id", "image_id", "token_ids"],
):
    for row in batch_arrow.to_pylist():
        TRAIN_CAPTIONS[int(row["image_id"])].append(
            (
                int(row["annotation_id"]),
                tuple(int(value) for value in row["token_ids"]),
            )
        )
assert len(TRAIN_CAPTIONS) == 113287
for image_id in TRAIN_CAPTIONS:
    TRAIN_CAPTIONS[image_id].sort()

VALIDATION_RECORDS = []
VALIDATION_REFERENCES = defaultdict(list)
for batch_arrow in pq.ParquetFile(VALIDATION_PARQUET).iter_batches(
    batch_size=8192,
    columns=["annotation_id", "image_id", "token_ids", "caption"],
):
    for row in batch_arrow.to_pylist():
        image_id = int(row["image_id"])
        VALIDATION_RECORDS.append(
            (
                int(row["annotation_id"]),
                image_id,
                tuple(int(value) for value in row["token_ids"]),
            )
        )
        VALIDATION_REFERENCES[image_id].append(row["caption"])
assert len(VALIDATION_RECORDS) == 25010
assert len(VALIDATION_REFERENCES) == 5000

TRAIN_IMAGE_IDS = tuple(sorted(TRAIN_PATHS))
VALIDATION_IMAGE_IDS = tuple(sorted(VALIDATION_PATHS))
SIGLIP_MEAN = torch.tensor(
    (0.5, 0.5, 0.5)
).view(3, 1, 1)
SIGLIP_STD = torch.tensor(
    (0.5, 0.5, 0.5)
).view(3, 1, 1)
IMAGENET_MEAN = torch.tensor(
    (0.485, 0.456, 0.406)
).view(3, 1, 1)
IMAGENET_STD = torch.tensor(
    (0.229, 0.224, 0.225)
).view(3, 1, 1)


def sample_seed(epoch, image_id):
    text = f"{SEED}|{epoch}|{image_id}|phase4p"
    return int.from_bytes(
        hashlib.sha256(text.encode("utf-8")).digest()[:8],
        "big",
    ) % (2 ** 63 - 1)


def shared_train_geometry(image, epoch, image_id):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(sample_seed(epoch, image_id))
        top, left, height, width = (
            transforms.RandomResizedCrop.get_params(
                image, scale=(0.80, 1.00), ratio=(0.75, 4 / 3)
            )
        )
        do_flip = bool(torch.rand(()) < 0.5)
    image = TF.resized_crop(
        image, top, left, height, width,
        size=[224, 224],
        interpolation=InterpolationMode.BICUBIC,
        antialias=True,
    )
    if do_flip:
        image = TF.hflip(image)
    return TF.to_tensor(image)


def validation_geometry(image):
    image = TF.resize(
        image, 256, interpolation=InterpolationMode.BICUBIC,
        antialias=True,
    )
    image = TF.center_crop(image, [224, 224])
    return TF.to_tensor(image)


def dual_normalize(base):
    return (
        (base - SIGLIP_MEAN) / SIGLIP_STD,
        (base - IMAGENET_MEAN) / IMAGENET_STD,
    )


class ProductionTrainDataset(Dataset):
    def __init__(self):
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(TRAIN_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = TRAIN_IMAGE_IDS[index]
        captions = TRAIN_CAPTIONS[image_id]
        offset = image_id % len(captions)
        annotation_id, token_ids = captions[
            (self.epoch + offset) % len(captions)
        ]
        with Image.open(TRAIN_PATHS[image_id]) as image:
            base = shared_train_geometry(
                image.convert("RGB"), self.epoch, image_id
            )
        teacher_image, student_image = dual_normalize(base)
        return {
            "image_id": image_id,
            "annotation_id": annotation_id,
            "teacher_image": teacher_image,
            "student_image": student_image,
            "token_ids": torch.tensor(token_ids, dtype=torch.long),
        }


class ValidationCaptionDataset(Dataset):
    def __len__(self):
        return len(VALIDATION_RECORDS)

    def __getitem__(self, index):
        annotation_id, image_id, token_ids = VALIDATION_RECORDS[index]
        with Image.open(VALIDATION_PATHS[image_id]) as image:
            base = validation_geometry(image.convert("RGB"))
        _, student_image = dual_normalize(base)
        return {
            "image_id": image_id,
            "annotation_id": annotation_id,
            "student_image": student_image,
            "token_ids": torch.tensor(token_ids, dtype=torch.long),
        }


class ValidationImageDataset(Dataset):
    def __len__(self):
        return len(VALIDATION_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = VALIDATION_IMAGE_IDS[index]
        with Image.open(VALIDATION_PATHS[image_id]) as image:
            base = validation_geometry(image.convert("RGB"))
        _, student_image = dual_normalize(base)
        return {"image_id": image_id, "student_image": student_image}


class EpochCursorSampler(Sampler):
    def __init__(self, length, epoch=0, cursor=0):
        self.length = int(length)
        self.epoch = int(epoch)
        self.cursor = int(cursor)

    def __iter__(self):
        generator = torch.Generator().manual_seed(
            SEED + self.epoch * 1000003
        )
        order = torch.randperm(
            self.length, generator=generator
        ).tolist()
        return iter(order[self.cursor:])

    def __len__(self):
        return self.length - self.cursor


def caption_collate(rows):
    maximum = max(row["token_ids"].numel() for row in rows)
    full = torch.zeros(len(rows), maximum, dtype=torch.long)
    for index, row in enumerate(rows):
        full[index, :row["token_ids"].numel()] = row["token_ids"]
    result = {
        "image_ids": torch.tensor(
            [row["image_id"] for row in rows], dtype=torch.long
        ),
        "annotation_ids": torch.tensor(
            [row["annotation_id"] for row in rows], dtype=torch.long
        ),
        "student_images": torch.stack(
            [row["student_image"] for row in rows]
        ),
        "decoder_input_ids": full[:, :-1],
        "labels": full[:, 1:],
    }
    result["decoder_attention_mask"] = (
        result["decoder_input_ids"].ne(0)
    )
    if "teacher_image" in rows[0]:
        result["teacher_images"] = torch.stack(
            [row["teacher_image"] for row in rows]
        )
    return result


def image_collate(rows):
    return {
        "image_ids": torch.tensor(
            [row["image_id"] for row in rows], dtype=torch.long
        ),
        "student_images": torch.stack(
            [row["student_image"] for row in rows]
        ),
    }


TRAIN_DATASET = ProductionTrainDataset()
VALIDATION_CAPTION_DATASET = ValidationCaptionDataset()
VALIDATION_IMAGE_DATASET = ValidationImageDataset()


def build_train_loader(epoch, cursor=0):
    TRAIN_DATASET.set_epoch(epoch)
    sampler = EpochCursorSampler(
        len(TRAIN_DATASET), epoch=epoch, cursor=cursor
    )
    return DataLoader(
        TRAIN_DATASET,
        batch_size=32,
        sampler=sampler,
        num_workers=2,
        pin_memory=True,
        persistent_workers=False,
        collate_fn=caption_collate,
    )


VALIDATION_CAPTION_LOADER = DataLoader(
    VALIDATION_CAPTION_DATASET,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=caption_collate,
)
VALIDATION_IMAGE_LOADER = DataLoader(
    VALIDATION_IMAGE_DATASET,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=image_collate,
)

loader_a = build_train_loader(0)
loader_b = build_train_loader(0)
batch_a = next(iter(loader_a))
batch_b = next(iter(loader_b))
for key in [
    "image_ids", "annotation_ids", "teacher_images",
    "student_images", "decoder_input_ids", "labels",
]:
    assert torch.equal(batch_a[key], batch_b[key])
assert len(set(batch_a["image_ids"].tolist())) == 32

write_json(
    AUDIT_DIR / "phase4p_cell02_data_pipeline.json",
    {
        "audit_version": "phase4p_cell02_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "train_images_per_epoch": len(TRAIN_DATASET),
        "captions_per_image_per_epoch": 1,
        "validation_captions": len(VALIDATION_CAPTION_DATASET),
        "validation_images": len(VALIDATION_IMAGE_DATASET),
        "physical_batch_size": 32,
        "shared_train_geometry": True,
        "stateless_augmentation": True,
        "two_loader_exact": True,
        "resume_cursor_supported": True,
        "validation_caption_text_read": True,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04P CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Train images / epoch      :", len(TRAIN_DATASET))
print("Captions / image / epoch  : 1")
print("Validation captions       :", len(VALIDATION_CAPTION_DATASET))
print("Validation images         :", len(VALIDATION_IMAGE_DATASET))
print("Physical/effective batch  : 32 / 64")
print("Shared train geometry     : True")
print("Stateless augmentation    : True")
print("Two-loader exact          : True")
print("Resume cursor supported   : True")
print("Test Set status           : SEALED")
print("Next                      : model and production optimizer")


PHASE 04P CELL 2 STATUS
Status                    : PASSED
Train images / epoch      : 113287
Captions / image / epoch  : 1
Validation captions       : 25010
Validation images         : 5000
Physical/effective batch  : 32 / 64
Shared train geometry     : True
Stateless augmentation    : True
Two-loader exact          : True
Resume cursor supported   : True
Test Set status           : SEALED
Next                      : model and production optimizer


## Cell 3 — Models, optimizer, scheduler và KD step engine

Strict-load Teacher C/S3, khóa Teacher frozen, dựng ba AdamW parameter
groups, warmup 5% + cosine, FP16 GradScaler, retry limit 10, gradient clip
1.0 và token-balanced accumulation.


In [3]:
CELL_STARTED_AT = time.perf_counter()
import timm
import torch.nn as nn
import torch.nn.functional as F
from torch.torch_version import TorchVersion
from transformers import SiglipModel


decoder_config = read_json(
    SHARED_DECODER_CONTRACT
)["architecture"]
assert int(decoder_config["vocabulary_size"]) == 10080
assert int(decoder_config["pad_id"]) == 0
assert int(decoder_config["maximum_decoder_steps"]) == 31


class SharedCaptionDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = dict(config)
        self.vocabulary_size = int(config["vocabulary_size"])
        self.pad_id = int(config["pad_id"])
        self.d_model = int(config["d_model"])
        self.maximum_decoder_steps = int(
            config["maximum_decoder_steps"]
        )
        self.token_embedding = nn.Embedding(
            self.vocabulary_size,
            self.d_model,
            padding_idx=self.pad_id,
        )
        self.position_embedding = nn.Embedding(
            self.maximum_decoder_steps, self.d_model
        )
        self.embedding_dropout = nn.Dropout(
            float(config["dropout"])
        )
        layer = nn.TransformerDecoderLayer(
            d_model=self.d_model,
            nhead=int(config["nhead"]),
            dim_feedforward=int(config["dim_feedforward"]),
            dropout=float(config["dropout"]),
            activation=config["activation"],
            batch_first=True,
            norm_first=True,
            bias=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            layer, num_layers=int(config["num_layers"])
        )
        self.final_norm = nn.LayerNorm(self.d_model)
        self.output_bias = nn.Parameter(
            torch.zeros(self.vocabulary_size)
        )

    def forward(
        self,
        decoder_input_ids,
        visual_memory,
        decoder_attention_mask=None,
    ):
        batch, length = decoder_input_ids.shape
        positions = torch.arange(
            length, device=decoder_input_ids.device
        ).unsqueeze(0).expand(batch, -1)
        target = (
            self.token_embedding(decoder_input_ids)
            * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        target = self.embedding_dropout(target)
        causal = torch.triu(
            torch.ones(
                length,
                length,
                dtype=torch.bool,
                device=decoder_input_ids.device,
            ),
            diagonal=1,
        )
        padding = (
            None
            if decoder_attention_mask is None
            else ~decoder_attention_mask.bool()
        )
        hidden = self.transformer_decoder(
            tgt=target,
            memory=visual_memory,
            tgt_mask=causal,
            tgt_key_padding_mask=padding,
        )
        hidden = self.final_norm(hidden)
        return F.linear(
            hidden,
            self.token_embedding.weight,
            self.output_bias,
        )


class TeacherVisualAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = nn.Linear(768, 256)
        self.position_embedding = nn.Parameter(
            torch.zeros(1, 196, 256)
        )
        self.output_norm = nn.LayerNorm(256)

    def forward(self, patches):
        assert patches.shape[1:] == (196, 768)
        return self.output_norm(
            self.projection(patches) + self.position_embedding
        )


def select_tensor_from_feature_output(feature_output):
    if torch.is_tensor(feature_output):
        return feature_output
    if isinstance(feature_output, dict):
        tensors = [
            value for value in feature_output.values()
            if torch.is_tensor(value)
        ]
        assert tensors
        return tensors[-1]
    if isinstance(feature_output, (tuple, list)):
        tensors = [
            value for value in feature_output
            if torch.is_tensor(value)
        ]
        assert tensors
        return tensors[-1]
    raise TypeError(type(feature_output).__name__)


class StudentVisualEncoderAdapter(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.projection = nn.Linear(320, 256)
        self.visual_position_embedding = nn.Parameter(
            torch.empty(1, 49, 256)
        )
        self.output_norm = nn.LayerNorm(256)

    def forward(self, images):
        feature = select_tensor_from_feature_output(
            self.backbone.forward_features(images)
        )
        assert tuple(feature.shape[1:]) == (320, 7, 7)
        tokens = feature.flatten(2).transpose(1, 2).contiguous()
        assert tokens.shape[1:] == (49, 320)
        return self.output_norm(
            self.projection(tokens)
            + self.visual_position_embedding
        )


os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
full_siglip = SiglipModel.from_pretrained(
    SIGLIP_SOURCE_DIR, local_files_only=True
)
TEACHER_ENCODER = full_siglip.vision_model
del full_siglip
TEACHER_ADAPTER = TeacherVisualAdapter()
TEACHER_DECODER = SharedCaptionDecoder(decoder_config)

tinyvit_backbone = timm.create_model(
    "tiny_vit_5m_224.dist_in22k_ft_in1k",
    pretrained=False,
)
if hasattr(tinyvit_backbone, "reset_classifier"):
    tinyvit_backbone.reset_classifier(
        num_classes=0, global_pool=""
    )
removed_heads = []
for head_name in ["head", "head_dist", "classifier", "fc"]:
    module = getattr(tinyvit_backbone, head_name, None)
    if (
        isinstance(module, nn.Module)
        and not isinstance(module, nn.Identity)
    ):
        setattr(tinyvit_backbone, head_name, nn.Identity())
        removed_heads.append(head_name)
assert removed_heads == ["head"]
STUDENT_ENCODER = StudentVisualEncoderAdapter(tinyvit_backbone)
STUDENT_DECODER = SharedCaptionDecoder(decoder_config)

with torch.serialization.safe_globals([TorchVersion]):
    teacher_checkpoint = torch.load(
        TEACHER_C_BEST_CHECKPOINT,
        map_location="cpu",
        weights_only=True,
    )
    student_encoder_checkpoint = torch.load(
        S3_INITIAL_CHECKPOINT,
        map_location="cpu",
        weights_only=True,
    )
    student_decoder_checkpoint = torch.load(
        SHARED_DECODER_INITIAL_CHECKPOINT,
        map_location="cpu",
        weights_only=True,
    )

assert teacher_checkpoint["schema_version"] == (
    "phase3_teacher_c_seed42_checkpoint_v1"
)
assert teacher_checkpoint["teacher_id"] == "C"
assert int(teacher_checkpoint["seed"]) == SEED
assert teacher_checkpoint["test_set_status"] == "sealed"
assert teacher_checkpoint["test_set_used"] is False

teacher_encoder_state = TEACHER_ENCODER.state_dict()
unknown_teacher_keys = set(
    teacher_checkpoint["encoder_candidate_state"]
) - set(teacher_encoder_state)
assert not unknown_teacher_keys, unknown_teacher_keys
teacher_encoder_state.update(
    teacher_checkpoint["encoder_candidate_state"]
)
TEACHER_ENCODER.load_state_dict(
    teacher_encoder_state, strict=True
)
teacher_adapter_load = TEACHER_ADAPTER.load_state_dict(
    teacher_checkpoint["adapter_state"], strict=True
)
teacher_decoder_load = TEACHER_DECODER.load_state_dict(
    teacher_checkpoint["decoder_state"], strict=True
)
student_encoder_load = STUDENT_ENCODER.load_state_dict(
    student_encoder_checkpoint["state_dict"], strict=True
)
student_decoder_load = STUDENT_DECODER.load_state_dict(
    student_decoder_checkpoint["state_dict"], strict=True
)
for result in [
    teacher_adapter_load,
    teacher_decoder_load,
    student_encoder_load,
    student_decoder_load,
]:
    assert not result.missing_keys
    assert not result.unexpected_keys

STUDENT_ENCODER_INITIAL_STATE = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_ENCODER.state_dict().items()
}
STUDENT_DECODER_INITIAL_STATE = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_DECODER.state_dict().items()
}

del teacher_checkpoint
del student_encoder_checkpoint
del student_decoder_checkpoint
del teacher_encoder_state

TEACHER_ENCODER = TEACHER_ENCODER.to(device)
TEACHER_ADAPTER = TEACHER_ADAPTER.to(device)
TEACHER_DECODER = TEACHER_DECODER.to(device)
STUDENT_ENCODER = STUDENT_ENCODER.to(device)
STUDENT_DECODER = STUDENT_DECODER.to(device)

for module in [
    TEACHER_ENCODER,
    TEACHER_ADAPTER,
    TEACHER_DECODER,
]:
    module.eval()
    for parameter in module.parameters():
        parameter.requires_grad_(False)


def teacher_forward(images, decoder_ids, decoder_mask):
    patches = TEACHER_ENCODER(
        pixel_values=images
    ).last_hidden_state
    assert patches.shape[1:] == (196, 768)
    memory = TEACHER_ADAPTER(patches)
    return TEACHER_DECODER(
        decoder_ids, memory, decoder_mask
    )


def student_forward(images, decoder_ids, decoder_mask):
    memory = STUDENT_ENCODER(images)
    assert memory.shape[1:] == (49, 256)
    return STUDENT_DECODER(
        decoder_ids, memory, decoder_mask
    )


def set_student_train_mode():
    STUDENT_ENCODER.train()
    STUDENT_DECODER.train()
    frozen_batchnorm = 0
    for module in STUDENT_ENCODER.modules():
        if isinstance(
            module, nn.modules.batchnorm._BatchNorm
        ):
            module.eval()
            frozen_batchnorm += 1
    assert frozen_batchnorm == 27
    return frozen_batchnorm


teacher_parameter_count = sum(
    parameter.numel()
    for module in [
        TEACHER_ENCODER,
        TEACHER_ADAPTER,
        TEACHER_DECODER,
    ]
    for parameter in module.parameters()
)
student_parameter_count = sum(
    parameter.numel()
    for module in [STUDENT_ENCODER, STUDENT_DECODER]
    for parameter in module.parameters()
)
assert student_parameter_count == 10_925_684
assert all(
    not parameter.requires_grad
    for module in [
        TEACHER_ENCODER,
        TEACHER_ADAPTER,
        TEACHER_DECODER,
    ]
    for parameter in module.parameters()
)


STUDENT_ENCODER_INITIAL = STUDENT_ENCODER_INITIAL_STATE
STUDENT_DECODER_INITIAL = STUDENT_DECODER_INITIAL_STATE


def set_student_train():
    return set_student_train_mode()


def teacher_logits(batch):
    return teacher_forward(
        batch["teacher_images"],
        batch["decoder_input_ids"],
        batch["decoder_attention_mask"],
    ).detach()


def student_logits(batch):
    return student_forward(
        batch["student_images"],
        batch["decoder_input_ids"],
        batch["decoder_attention_mask"],
    )

TEMPERATURE = 4.0
ALPHA = 0.5
BACKWARD_DIVISOR = 1024.0
ACCUMULATION = 2
STEPS_PER_EPOCH = math.ceil(113287 / 64)
MAX_EPOCHS = 20
TOTAL_STEPS = STEPS_PER_EPOCH * MAX_EPOCHS
WARMUP_STEPS = round(TOTAL_STEPS * 0.05)

backbone_parameters = list(STUDENT_ENCODER.backbone.parameters())
backbone_ids = {id(parameter) for parameter in backbone_parameters}
adapter_parameters = [
    parameter for parameter in STUDENT_ENCODER.parameters()
    if id(parameter) not in backbone_ids
]
decoder_parameters = list(STUDENT_DECODER.parameters())


def build_optimizer_state():
    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_parameters, "lr": 1e-4},
            {"params": adapter_parameters, "lr": 3e-4},
            {"params": decoder_parameters, "lr": 3e-4},
        ],
        betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-4,
    )

    def factor(step):
        if step < WARMUP_STEPS:
            return (step + 1) / WARMUP_STEPS
        progress = (
            (step - WARMUP_STEPS)
            / max(1, TOTAL_STEPS - WARMUP_STEPS)
        )
        return 0.05 + 0.95 * 0.5 * (
            1.0 + math.cos(math.pi * min(progress, 1.0))
        )

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=factor
    )
    scaler = torch.amp.GradScaler(
        "cuda", enabled=True, init_scale=1024.0
    )
    return optimizer, scheduler, scaler


def loss_sums(student, teacher, labels):
    valid = labels.ne(0)
    valid_count = int(valid.sum())
    ce_sum = F.cross_entropy(
        student.float().reshape(-1, 10080),
        labels.reshape(-1),
        ignore_index=0,
        reduction="sum",
    )
    student_log_probs = F.log_softmax(
        student.float() / TEMPERATURE, dim=-1
    )
    teacher_probs = F.softmax(
        teacher.float() / TEMPERATURE, dim=-1
    )
    kl_sum = (
        F.kl_div(
            student_log_probs, teacher_probs, reduction="none"
        ).sum(dim=-1)
        * valid
    ).sum() * (TEMPERATURE ** 2)
    combined_sum = (1 - ALPHA) * ce_sum + ALPHA * kl_sum
    return combined_sum, ce_sum, kl_sum, valid_count


def to_device(batch):
    return {
        key: (
            value.to(device, non_blocking=True)
            if isinstance(value, torch.Tensor) else value
        )
        for key, value in batch.items()
    }


def optimizer_step(
    microbatches, optimizer, scheduler, scaler,
    retry_limit=10,
):
    retries = 0
    while True:
        set_student_train()
        optimizer.zero_grad(set_to_none=True)
        combined_total = 0.0
        ce_total = 0.0
        kd_total = 0.0
        valid_total = 0
        for cpu_batch in microbatches:
            current = to_device(cpu_batch)
            with torch.amp.autocast(
                "cuda", dtype=torch.float16
            ):
                teacher = teacher_logits(current)
                student = student_logits(current)
                combined, ce_sum, kd_sum, valid = loss_sums(
                    student, teacher, current["labels"]
                )
            scaler.scale(
                combined / BACKWARD_DIVISOR
            ).backward()
            combined_total += float(combined.detach())
            ce_total += float(ce_sum.detach())
            kd_total += float(kd_sum.detach())
            valid_total += valid
        scaler.unscale_(optimizer)
        correction = BACKWARD_DIVISOR / valid_total
        for parameter in (
            list(STUDENT_ENCODER.parameters())
            + list(STUDENT_DECODER.parameters())
        ):
            if parameter.grad is not None:
                parameter.grad.mul_(correction)
        gradient_norm = torch.nn.utils.clip_grad_norm_(
            list(STUDENT_ENCODER.parameters())
            + list(STUDENT_DECODER.parameters()),
            1.0,
        )
        old_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        new_scale = scaler.get_scale()
        if new_scale < old_scale:
            retries += 1
            assert retries <= retry_limit
            continue
        assert torch.isfinite(gradient_norm)
        scheduler.step()
        return {
            "combined_loss": combined_total / valid_total,
            "ce": ce_total / valid_total,
            "kd_kl_t2": kd_total / valid_total,
            "valid_tokens": valid_total,
            "gradient_norm": float(gradient_norm),
            "overflow_retries": retries,
            "scaler": float(new_scale),
        }


optimizer, scheduler, scaler = build_optimizer_state()
assert STEPS_PER_EPOCH == 1771
assert TOTAL_STEPS == 35420
assert WARMUP_STEPS == 1771

write_json(
    CONFIG_DIR / "phase4p_production_optimizer_contract.json",
    {
        "contract_version": "phase4p_optimizer_v1",
        "created_at_utc": now_utc(),
        "status": "locked",
        "optimizer": "AdamW",
        "learning_rates": {
            "backbone": 1e-4, "adapter": 3e-4, "decoder": 3e-4,
        },
        "weight_decay": 1e-4,
        "physical_batch": 32,
        "accumulation": 2,
        "effective_batch": 64,
        "steps_per_epoch": STEPS_PER_EPOCH,
        "maximum_epochs": MAX_EPOCHS,
        "total_steps": TOTAL_STEPS,
        "warmup_steps": WARMUP_STEPS,
        "minimum_lr_factor": 0.05,
        "mixed_precision": "FP16 plus GradScaler",
        "gradscaler_initial_scale": 1024.0,
        "overflow_retry_limit": 10,
        "backward_divisor": BACKWARD_DIVISOR,
        "gradient_clip": 1.0,
        "teacher_frozen": True,
        "temperature": TEMPERATURE,
        "alpha": ALPHA,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
write_json(
    AUDIT_DIR / "phase4p_cell03_model_optimizer.json",
    {
        "audit_version": "phase4p_cell03_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "strict_loads": 5,
        "teacher_frozen": True,
        "student_parameters": 10925684,
        "optimizer_parameter_groups": 3,
        "steps_per_epoch": STEPS_PER_EPOCH,
        "total_steps": TOTAL_STEPS,
        "scheduler_advanced": False,
        "optimizer_step_performed": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04P CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Strict checkpoint loads   : 5 / 5 PASSED")
print("Teacher frozen            : True")
print("Student parameters        : 10925684")
print("Optimizer groups          : 3 / 3")
print("Physical/effective batch  : 32 / 64")
print("Steps / epoch             :", STEPS_PER_EPOCH)
print("Total optimizer steps     :", TOTAL_STEPS)
print("Warmup steps              :", WARMUP_STEPS)
print("FP16 / GradScaler         : ENABLED / 1024")
print("Overflow retry limit      : 10")
print("Token-balanced accumulation: READY")
print("Optimizer step performed  : False")
print("Test Set status           : SEALED")
print("Next                      : Validation and Beam-3 engine")


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

PHASE 04P CELL 3 STATUS
Status                    : PASSED
Strict checkpoint loads   : 5 / 5 PASSED
Teacher frozen            : True
Student parameters        : 10925684
Optimizer groups          : 3 / 3
Physical/effective batch  : 32 / 64
Steps / epoch             : 1771
Total optimizer steps     : 35420
Warmup steps              : 1771
FP16 / GradScaler         : ENABLED / 1024
Overflow retry limit      : 10
Token-balanced accumulation: READY
Optimizer step performed  : False
Test Set status           : SEALED
Next                      : Validation and Beam-3 engine


## Cell 4 — Offline COCO metrics và Validation/Beam-3

Khóa teacher-forced Validation, Beam-3 generation trên 5.000 ảnh và backend
BLEU-1/BLEU-4/METEOR/ROUGE-L/CIDEr-D. Không mở Held-out Test.


In [4]:
CELL_STARTED_AT = time.perf_counter()
try:
    from pycocoevalcap.bleu.bleu import Bleu
except Exception:
    wheel_matches = sorted(
        INPUT_ROOT.rglob("pycocoevalcap-1.2-py3-none-any.whl")
    )
    assert len(wheel_matches) == 1, wheel_matches
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-index", str(wheel_matches[0]),
        ],
        check=True,
    )
    from pycocoevalcap.bleu.bleu import Bleu

from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer


VOCAB_PATH = exactly_one("word_vocabulary_minfreq5.json")
vocabulary_payload = read_json(VOCAB_PATH)
if "id_to_token" in vocabulary_payload:
    raw_id_to_token = vocabulary_payload["id_to_token"]
    if isinstance(raw_id_to_token, list):
        ID_TO_TOKEN = {
            index: token
            for index, token in enumerate(raw_id_to_token)
        }
    else:
        ID_TO_TOKEN = {
            int(key): value
            for key, value in raw_id_to_token.items()
        }
else:
    token_to_id = vocabulary_payload["token_to_id"]
    ID_TO_TOKEN = {int(value): key for key, value in token_to_id.items()}


def decode_tokens(token_ids):
    words = []
    for token_id in token_ids:
        token_id = int(token_id)
        if token_id == 2:
            break
        if token_id in {0, 1, 3}:
            continue
        words.append(ID_TO_TOKEN[token_id])
    return " ".join(words)


def compute_coco_metrics(references, hypotheses):
    ground_truth = {
        int(image_id): [
            {"caption": caption} for caption in captions
        ]
        for image_id, captions in references.items()
    }
    predictions = {
        int(image_id): [{"caption": hypotheses[int(image_id)]}]
        for image_id in references
    }
    tokenizer = PTBTokenizer()
    tokenized_gt = tokenizer.tokenize(ground_truth)
    tokenized_pred = tokenizer.tokenize(predictions)
    bleu, _ = Bleu(4).compute_score(
        tokenized_gt, tokenized_pred
    )
    meteor, _ = Meteor().compute_score(
        tokenized_gt, tokenized_pred
    )
    rouge, _ = Rouge().compute_score(
        tokenized_gt, tokenized_pred
    )
    cider, _ = Cider().compute_score(
        tokenized_gt, tokenized_pred
    )
    return {
        "BLEU-1": float(bleu[0]),
        "BLEU-4": float(bleu[3]),
        "METEOR": float(meteor),
        "ROUGE-L": float(rouge),
        "CIDEr-D": float(cider),
    }


smoke_ids = list(VALIDATION_IMAGE_IDS[:32])
smoke_references = {
    image_id: VALIDATION_REFERENCES[image_id]
    for image_id in smoke_ids
}
smoke_hypotheses = {
    image_id: VALIDATION_REFERENCES[image_id][0]
    for image_id in smoke_ids
}
metric_smoke = compute_coco_metrics(
    smoke_references, smoke_hypotheses
)
assert all(math.isfinite(value) for value in metric_smoke.values())


def teacher_forced_ce(loader, maximum_batches=None):
    STUDENT_ENCODER.eval()
    STUDENT_DECODER.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for batch_index, cpu_batch in enumerate(loader):
            current = to_device(cpu_batch)
            with torch.amp.autocast(
                "cuda", dtype=torch.float16
            ):
                logits = student_logits(current)
            valid = current["labels"].ne(0)
            loss_sum = F.cross_entropy(
                logits.float().reshape(-1, 10080),
                current["labels"].reshape(-1),
                ignore_index=0,
                reduction="sum",
            )
            total_loss += float(loss_sum)
            total_tokens += int(valid.sum())
            if (
                maximum_batches is not None
                and batch_index + 1 >= maximum_batches
            ):
                break
    return total_loss / total_tokens, total_tokens


def beam_generate_one(student_image, beam_size=3):
    STUDENT_ENCODER.eval()
    STUDENT_DECODER.eval()
    with torch.no_grad():
        memory = STUDENT_ENCODER(
            student_image.unsqueeze(0).to(device)
        )
        beams = [([1], 0.0, False)]
        for step in range(31):
            candidates = []
            active = [item for item in beams if not item[2]]
            if not active:
                break
            for tokens, score, finished in beams:
                if finished:
                    candidates.append((tokens, score, True))
                    continue
                ids = torch.tensor(
                    tokens, dtype=torch.long, device=device
                ).unsqueeze(0)
                mask = ids.ne(0)
                logits = STUDENT_DECODER(ids, memory, mask)[:, -1]
                log_probs = F.log_softmax(logits.float(), dim=-1)[0]
                log_probs[[0, 1, 3]] = -torch.inf
                lexical_count = max(0, len(tokens) - 1)
                if lexical_count < 2:
                    log_probs[2] = -torch.inf
                values, indices = torch.topk(log_probs, beam_size)
                for value, index in zip(
                    values.tolist(), indices.tolist()
                ):
                    next_tokens = tokens + [index]
                    candidates.append(
                        (
                            next_tokens,
                            score + value,
                            index == 2,
                        )
                    )
            def rank(item):
                length = max(1, len(item[0]) - 1)
                return item[1] / (length ** 0.7)
            beams = sorted(
                candidates, key=rank, reverse=True
            )[:beam_size]
            if all(item[2] for item in beams):
                break
        best = max(
            beams,
            key=lambda item: item[1] / (
                max(1, len(item[0]) - 1) ** 0.7
            ),
        )
        return best[0], best[1]


validation_smoke_batch = next(iter(VALIDATION_IMAGE_LOADER))
tokens_a, score_a = beam_generate_one(
    validation_smoke_batch["student_images"][0]
)
tokens_b, score_b = beam_generate_one(
    validation_smoke_batch["student_images"][0]
)
assert tokens_a == tokens_b
assert score_a == score_b
smoke_ce, smoke_tokens = teacher_forced_ce(
    VALIDATION_CAPTION_LOADER, maximum_batches=1
)
assert math.isfinite(smoke_ce)

REFERENCES_PATH = (
    VALIDATION_DIR / "karpathy_validation_references.json"
)
write_json(
    REFERENCES_PATH,
    {
        str(image_id): captions
        for image_id, captions in VALIDATION_REFERENCES.items()
    },
)
write_json(
    AUDIT_DIR / "phase4p_cell04_validation_engine.json",
    {
        "audit_version": "phase4p_cell04_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "validation_images": 5000,
        "validation_references": 25010,
        "metric_backend": "pycocoevalcap 1.2",
        "metric_smoke": metric_smoke,
        "teacher_forced_smoke_ce": smoke_ce,
        "teacher_forced_smoke_tokens": smoke_tokens,
        "beam_size": 3,
        "length_penalty": 0.7,
        "beam_repeat_exact": True,
        "full_validation_run": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04P CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Validation images         : 5000")
print("Validation references     : 25010")
print("COCO metrics              : 5 / 5 WIRED")
print("Teacher-forced CE smoke   :", smoke_ce)
print("Beam size / penalty       : 3 / 0.7")
print("Beam repeat exact         : True")
print("Full Validation run       : False")
print("Test Set status           : SEALED")
print("Next                      : atomic checkpoint and production pilot")


Processing /kaggle/input/datasets/nguynthnhdan/pycocoevalcap-1-2/pycocoevalcap-1.2-py3-none-any.whl


PTBTokenizer tokenized 1986 tokens at 18695.38 tokens per second.
PTBTokenizer tokenized 385 tokens at 6551.97 tokens per second.


{'testlen': 328, 'reflen': 328, 'guess': [328, 296, 264, 232], 'correct': [328, 296, 264, 232]}
ratio: 0.9999999999969512
PHASE 04P CELL 4 STATUS
Status                    : PASSED
Validation images         : 5000
Validation references     : 25010
COCO metrics              : 5 / 5 WIRED
Teacher-forced CE smoke   : 11.347428206559066
Beam size / penalty       : 3 / 0.7
Beam repeat exact         : True
Full Validation run       : False
Test Set status           : SEALED
Next                      : atomic checkpoint and production pilot


## Cell 5 — Atomic checkpoint và discarded production pilot

Chạy bốn optimizer step trên 256 ảnh, kiểm tra atomic save/load, state exact,
xóa checkpoint tạm và khôi phục initialization. Đây không phải full training.


In [5]:
CELL_STARTED_AT = time.perf_counter()


def clone_cpu(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {key: clone_cpu(item) for key, item in value.items()}
    if isinstance(value, list):
        return [clone_cpu(item) for item in value]
    if isinstance(value, tuple):
        return tuple(clone_cpu(item) for item in value)
    return copy.deepcopy(value)


def nested_exact(left, right):
    if isinstance(left, torch.Tensor):
        return (
            isinstance(right, torch.Tensor)
            and left.dtype == right.dtype
            and left.shape == right.shape
            and torch.equal(left.cpu(), right.cpu())
        )
    if isinstance(left, dict):
        return (
            isinstance(right, dict)
            and left.keys() == right.keys()
            and all(nested_exact(left[key], right[key]) for key in left)
        )
    if isinstance(left, (list, tuple)):
        return (
            isinstance(right, type(left))
            and len(left) == len(right)
            and all(nested_exact(a, b) for a, b in zip(left, right))
        )
    return left == right


def restore_student_initial():
    STUDENT_ENCODER.load_state_dict(
        STUDENT_ENCODER_INITIAL, strict=True
    )
    STUDENT_DECODER.load_state_dict(
        STUDENT_DECODER_INITIAL, strict=True
    )
    for module in [STUDENT_ENCODER, STUDENT_DECODER]:
        module.zero_grad(set_to_none=True)


def atomic_save(path, payload):
    temporary = Path(str(path) + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def save_training_checkpoint(
    path, optimizer, scheduler, scaler, progress
):
    atomic_save(
        path,
        {
            "schema_version": "phase4_teacher_c_s3_kd_checkpoint_v1",
            "experiment_id": "P4C_C_S3_LOGIT",
            "student_encoder_state": clone_cpu(
                STUDENT_ENCODER.state_dict()
            ),
            "student_decoder_state": clone_cpu(
                STUDENT_DECODER.state_dict()
            ),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "progress": progress,
            "cpu_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all(),
            "teacher_checkpoint_sha256": sha256_file(
                TEACHER_CHECKPOINT
            ),
            "temperature": TEMPERATURE,
            "alpha": ALPHA,
            "test_set_status": "sealed",
            "test_set_used": False,
        },
    )


restore_student_initial()
optimizer, scheduler, scaler = build_optimizer_state()
pilot_loader = build_train_loader(epoch=0, cursor=0)
pilot_iterator = iter(pilot_loader)
pilot_rows = []
pilot_started = time.perf_counter()
cursor = 0
overflow_total = 0
for optimizer_step_index in range(1, 5):
    microbatches = [
        next(pilot_iterator) for _ in range(ACCUMULATION)
    ]
    result = optimizer_step(
        microbatches, optimizer, scheduler, scaler,
        retry_limit=10,
    )
    cursor += sum(len(batch["image_ids"]) for batch in microbatches)
    overflow_total += result["overflow_retries"]
    row = {
        "optimizer_step": optimizer_step_index,
        "cursor": cursor,
        **result,
        "lr_backbone": optimizer.param_groups[0]["lr"],
        "lr_adapter": optimizer.param_groups[1]["lr"],
        "lr_decoder": optimizer.param_groups[2]["lr"],
    }
    pilot_rows.append(row)
    print(
        f"step={optimizer_step_index:02d} | cursor={cursor} | "
        f"loss={result['combined_loss']:.6f} | "
        f"ce={result['ce']:.6f} | kd={result['kd_kl_t2']:.6f} | "
        f"grad={result['gradient_norm']:.4f}"
    )
pilot_seconds = time.perf_counter() - pilot_started
assert cursor == 256
assert all(math.isfinite(row["combined_loss"]) for row in pilot_rows)

TEMP_CHECKPOINT = CHECKPOINT_DIR / "temporary_production_pilot.pt"
progress = {
    "epoch": 0,
    "cursor": cursor,
    "global_optimizer_step": 4,
}
save_training_checkpoint(
    TEMP_CHECKPOINT, optimizer, scheduler, scaler, progress
)
expected_encoder = clone_cpu(STUDENT_ENCODER.state_dict())
expected_decoder = clone_cpu(STUDENT_DECODER.state_dict())
expected_optimizer = clone_cpu(optimizer.state_dict())
expected_scheduler = clone_cpu(scheduler.state_dict())
expected_scaler = clone_cpu(scaler.state_dict())
expected_cpu_rng = torch.get_rng_state().clone()
expected_cuda_rng = [
    state.clone() for state in torch.cuda.get_rng_state_all()
]

restore_student_initial()
loaded_optimizer, loaded_scheduler, loaded_scaler = (
    build_optimizer_state()
)
with torch.serialization.safe_globals([TorchVersion]):
    checkpoint = torch.load(
        TEMP_CHECKPOINT, map_location="cpu", weights_only=True
    )
STUDENT_ENCODER.load_state_dict(
    checkpoint["student_encoder_state"], strict=True
)
STUDENT_DECODER.load_state_dict(
    checkpoint["student_decoder_state"], strict=True
)
loaded_optimizer.load_state_dict(checkpoint["optimizer_state"])
loaded_scheduler.load_state_dict(checkpoint["scheduler_state"])
loaded_scaler.load_state_dict(checkpoint["scaler_state"])
torch.set_rng_state(checkpoint["cpu_rng"])
torch.cuda.set_rng_state_all(checkpoint["cuda_rng"])

assert nested_exact(expected_encoder, STUDENT_ENCODER.state_dict())
assert nested_exact(expected_decoder, STUDENT_DECODER.state_dict())
assert nested_exact(expected_optimizer, loaded_optimizer.state_dict())
assert nested_exact(expected_scheduler, loaded_scheduler.state_dict())
assert nested_exact(expected_scaler, loaded_scaler.state_dict())
assert torch.equal(expected_cpu_rng, torch.get_rng_state())
assert all(
    torch.equal(left, right)
    for left, right in zip(
        expected_cuda_rng, torch.cuda.get_rng_state_all()
    )
)
TEMP_CHECKPOINT.unlink()
assert not TEMP_CHECKPOINT.exists()
restore_student_initial()

PILOT_CURVE = LOG_DIR / "phase4p_discarded_pilot.csv"
with PILOT_CURVE.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle, fieldnames=list(pilot_rows[0])
    )
    writer.writeheader()
    writer.writerows(pilot_rows)

seconds_per_step = pilot_seconds / 4
estimated_minutes_per_epoch = (
    seconds_per_step * STEPS_PER_EPOCH / 60
)
write_json(
    AUDIT_DIR / "phase4p_cell05_production_pilot.json",
    {
        "audit_version": "phase4p_cell05_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "optimizer_steps": 4,
        "microbatches": 8,
        "images": cursor,
        "overflow_retries": overflow_total,
        "seconds_per_optimizer_step": seconds_per_step,
        "estimated_minutes_per_epoch": estimated_minutes_per_epoch,
        "atomic_checkpoint": True,
        "resume_load_exact": True,
        "temporary_checkpoint_deleted": True,
        "student_initial_restore_exact": True,
        "teacher_frozen": True,
        "pilot_weights_saved": False,
        "full_validation_run": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04P CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Pilot optimizer steps     : 4 / 4")
print("Pilot images              :", cursor)
print("Overflow retries          :", overflow_total)
print("Seconds / optimizer step  :", round(seconds_per_step, 4))
print("Estimated min / epoch     :", round(estimated_minutes_per_epoch, 2))
print("Atomic checkpoint         : PASSED")
print("Resume load exact         : True")
print("Temporary checkpoint      : DELETED")
print("Initial restore exact     : True")
print("Pilot weights saved       : False")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Next                      : lock production run contract")


step=01 | cursor=64 | loss=7.856521 | ce=11.092940 | kd=4.620102 | grad=2.7286
step=02 | cursor=128 | loss=7.894489 | ce=11.121873 | kd=4.667105 | grad=2.9396
step=03 | cursor=192 | loss=7.850337 | ce=11.038882 | kd=4.661792 | grad=3.1336
step=04 | cursor=256 | loss=7.933806 | ce=11.095751 | kd=4.771860 | grad=3.0780
PHASE 04P CELL 5 STATUS
Status                    : PASSED
Pilot optimizer steps     : 4 / 4
Pilot images              : 256
Overflow retries          : 0
Seconds / optimizer step  : 2.0139
Estimated min / epoch     : 59.44
Atomic checkpoint         : PASSED
Resume load exact         : True
Temporary checkpoint      : DELETED
Initial restore exact     : True
Pilot weights saved       : False
Full training started     : False
Test Set status           : SEALED
Next                      : lock production run contract


## Cell 6 — Lock production run contract

Khóa toàn bộ dữ liệu, mô hình, KD loss, optimizer, Validation, Beam-3,
checkpoint, resume và early-stopping cho full run seed 42.


In [6]:
CELL_STARTED_AT = time.perf_counter()
PRODUCTION_CONTRACT = (
    CONFIG_DIR / "phase4p_teacher_c_s3_kd_production_contract.json"
)
production_contract = {
    "contract_version": "phase4p_teacher_c_s3_kd_production_v1",
    "created_at_utc": now_utc(),
    "status": "locked",
    "experiment_id": "P4C_C_S3_LOGIT",
    "teacher": {
        "id": "C",
        "seed": 42,
        "checkpoint": str(TEACHER_CHECKPOINT),
        "checkpoint_sha256": sha256_file(TEACHER_CHECKPOINT),
        "mode": "eval",
        "frozen": True,
    },
    "student": {
        "id": "S3",
        "seed": 42,
        "initial_encoder_checkpoint": str(S3_INITIAL_CHECKPOINT),
        "initial_decoder_checkpoint": str(
            DECODER_INITIAL_CHECKPOINT
        ),
    },
    "kd": {
        "channel": "direct_logit",
        "temperature": TEMPERATURE,
        "alpha": ALPHA,
        "padding_mask": "labels != PAD_ID",
        "teacher_logits_detached": True,
    },
    "training": {
        "images_per_epoch": 113287,
        "caption_sampling": "one per image per epoch cyclic",
        "physical_batch": 32,
        "accumulation": 2,
        "effective_batch": 64,
        "maximum_epochs": 20,
        "optimizer": "AdamW",
        "scheduler": "warmup 5 percent plus cosine",
        "mixed_precision": "FP16 plus GradScaler",
        "gradient_clip": 1.0,
        "overflow_retry_limit": 10,
        "atomic_last_every_optimizer_steps": 250,
        "resume_boundary": "optimizer step",
    },
    "validation": {
        "caption_count": 25010,
        "image_count": 5000,
        "generation": "Beam-3 length penalty 0.7",
        "metrics": [
            "BLEU-1", "BLEU-4", "METEOR", "ROUGE-L", "CIDEr-D"
        ],
        "selection_primary": "CIDEr-D",
        "tie_break": ["BLEU-4", "CE", "earlier epoch"],
    },
    "early_stopping": {
        "minimum_epoch": 5,
        "patience": 4,
        "minimum_delta": 0.001,
    },
    "comparison": {
        "baseline": "S3 CE-only seed42",
        "teacher_metric_is_not_kd_improvement": True,
    },
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(PRODUCTION_CONTRACT, production_contract)
write_json(
    AUDIT_DIR / "phase4p_cell06_production_contract.json",
    {
        "audit_version": "phase4p_cell06_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "contract": str(PRODUCTION_CONTRACT),
        "contract_sha256": sha256_file(PRODUCTION_CONTRACT),
        "selection_split": "validation",
        "primary_metric": "CIDEr-D",
        "atomic_last_checkpoint": True,
        "atomic_best_checkpoint": True,
        "mid_epoch_resume": True,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04P CELL 6 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Production contract       : LOCKED")
print("Maximum epochs            : 20")
print("Teacher checkpoint        : LOCKED")
print("Student initialization    : LOCKED")
print("Selection split           : Validation")
print("Primary metric            : CIDEr-D")
print("Beam size / penalty       : 3 / 0.7")
print("Atomic last / best        : READY / READY")
print("Mid-epoch resume          : READY")
print("Early stopping            : epoch>=5 / patience=4")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Next                      : package and authorize Phase 04Q")


PHASE 04P CELL 6 STATUS
Status                    : PASSED
Production contract       : LOCKED
Maximum epochs            : 20
Teacher checkpoint        : LOCKED
Student initialization    : LOCKED
Selection split           : Validation
Primary metric            : CIDEr-D
Beam size / penalty       : 3 / 0.7
Atomic last / best        : READY / READY
Mid-epoch resume          : READY
Early stopping            : epoch>=5 / patience=4
Full training started     : False
Test Set status           : SEALED
Next                      : package and authorize Phase 04Q


## Cell 7 — Package Phase 04P và authorize Phase 04Q

Tạo manifest/checksum, completion marker và authorization cho full
Teacher C→S3 Direct Logit-KD seed-42 run.


In [7]:
CELL_STARTED_AT = time.perf_counter()
MANIFEST_CSV = MANIFEST_DIR / "phase4p_manifest.csv"
MANIFEST_JSON = MANIFEST_DIR / "phase4p_manifest.json"
SHA256SUMS = MANIFEST_DIR / "phase4p_SHA256SUMS.txt"
COMPLETE_MARKER = OUTPUT_ROOT / (
    "PHASE4_TEACHER_C_S3_LOGIT_KD_TRAINING_READINESS_COMPLETE.json"
)
AUTHORIZED_MARKER = OUTPUT_ROOT / (
    "PHASE4_TEACHER_C_S3_LOGIT_KD_SEED42_FULL_RUN_AUTHORIZED.json"
)

write_json(
    AUTHORIZED_MARKER,
    {
        "marker_version": (
            "phase4_teacher_c_s3_logit_kd_seed42_"
            "full_run_authorized_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "authorized",
        "experiment_id": "P4C_C_S3_LOGIT",
        "seed": 42,
        "production_contract": {
            "relative_path": PRODUCTION_CONTRACT.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(PRODUCTION_CONTRACT),
        },
        "next_phase": (
            "Phase 04Q Teacher C to S3 logit KD seed42 full run"
        ),
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

artifact_paths = sorted(
    [
        path for path in OUTPUT_ROOT.rglob("*")
        if path.is_file()
        and path not in {
            MANIFEST_CSV, MANIFEST_JSON, SHA256SUMS, COMPLETE_MARKER
        }
        and not path.name.endswith(".tmp")
    ],
    key=lambda path: path.relative_to(OUTPUT_ROOT).as_posix(),
)
manifest_rows = [
    {
        "relative_path": path.relative_to(OUTPUT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in artifact_paths
]
with MANIFEST_CSV.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
write_json(
    MANIFEST_JSON,
    {
        "schema_version": "phase4p_manifest_v1",
        "created_at_utc": now_utc(),
        "artifact_count": len(manifest_rows),
        "artifacts": manifest_rows,
    },
)
SHA256SUMS.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)
for row in manifest_rows:
    path = resolve_inside(OUTPUT_ROOT, row["relative_path"])
    assert path.stat().st_size == int(row["size_bytes"])
    assert sha256_file(path) == row["sha256"]

write_json(
    COMPLETE_MARKER,
    {
        "marker_version": (
            "phase4_teacher_c_s3_logit_kd_"
            "training_readiness_complete_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "complete",
        "experiment_id": "P4C_C_S3_LOGIT",
        "seed": 42,
        "production_data_pipeline": "passed",
        "model_optimizer_engine": "passed",
        "validation_beam_engine": "passed",
        "discarded_pilot": "passed",
        "production_contract": "locked",
        "phase04q_full_run_authorized": True,
        "authorization_marker": {
            "relative_path": AUTHORIZED_MARKER.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(AUTHORIZED_MARKER),
        },
        "manifest": {
            "csv": {
                "relative_path": MANIFEST_CSV.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_CSV),
            },
            "json": {
                "relative_path": MANIFEST_JSON.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_JSON),
            },
            "sha256sums": {
                "relative_path": SHA256SUMS.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(SHA256SUMS),
            },
        },
        "manifest_rows": len(manifest_rows),
        "manifest_verification": "all_passed",
        "pilot_optimizer_steps_only": True,
        "pilot_weights_saved": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

print("=" * 72)
print("PHASE 04P CELL 7 STATUS")
print("=" * 72)
print("Status                    : COMPLETE")
print("Experiment                : Teacher C -> S3 logit KD")
print("Production data pipeline  : PASSED")
print("Model/optimizer engine    : PASSED")
print("Validation/Beam-3 engine  : PASSED")
print("Discarded pilot           : 4 / 4 PASSED")
print("Production contract       : LOCKED")
print("Manifest rows             :", len(manifest_rows))
print("Manifest verification     : ALL PASSED")
print("Phase 04Q full seed42 run : AUTHORIZED")
print("Pilot weights saved       : False")
print("Full training started     : False")
print("Test Set status           : SEALED")
print(
    "Elapsed                   :",
    round(time.perf_counter() - CELL_STARTED_AT, 2),
    "giây",
)
print("Next                      : Save Version, then Phase 04Q")


PHASE 04P CELL 7 STATUS
Status                    : COMPLETE
Experiment                : Teacher C -> S3 logit KD
Production data pipeline  : PASSED
Model/optimizer engine    : PASSED
Validation/Beam-3 engine  : PASSED
Discarded pilot           : 4 / 4 PASSED
Production contract       : LOCKED
Manifest rows             : 11
Manifest verification     : ALL PASSED
Phase 04Q full seed42 run : AUTHORIZED
Pilot weights saved       : False
Full training started     : False
Test Set status           : SEALED
Elapsed                   : 0.02 giây
Next                      : Save Version, then Phase 04Q
